# 05. Backtesting walk-forward (métricas online)

**Proyecto**: Pronóstico de demanda multi-series (Curso II — Especialización ML Engineering)

Completa la **Fase 3** con la evaluación **online**: un backtesting walk-forward que simula
producción sobre el historial (hasta 2017-09-30).

- Se divide el pasado en **4 ventanas de ~90 días** (folds): dic-2016, mar-2017, jun-2017 y sep-2017.
- Por cada fold: se entrena con **todo lo anterior** a la ventana y se predice la ventana
  (exactamente lo que haría el sistema en producción cada trimestre).
- Se acumulan las métricas de todos los folds → **métricas online** (promedio entre folds).

Sin fuga de datos: las features se construyen sobre el historial completo (lags/rolling solo
usan valores pasados) y el denominador del MASE se calcula sobre el train de cada fold.

> Código reusable: `src/models/backtesting.py`. Los resultados también se registran en MLflow
> (experimento `demand_forecast_backtest`) como evidencia del pipeline online.

In [1]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

# Localiza la raíz del proyecto estés donde estés
def _find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw" / "train.csv").exists():
            return p
    raise FileNotFoundError("No se encontró la raíz del proyecto (data/raw/train.csv)")

ROOT = _find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

from src.features.build_features import FEATURE_COLUMNS  # noqa: E402
from src.models.backtesting import (  # noqa: E402
    DEFAULT_TEST_ENDS,
    backtest_to_mlflow,
    walk_forward_backtest,
    walk_forward_folds,
)
from src.models.configs import all_configs  # noqa: E402

sns.set_theme()
pd.set_option("display.float_format", "{:.2f}".format)

DATA_OUT = ROOT / "data" / "processed"
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
TRACKING_URI = f"sqlite:///{(ROOT / 'mlruns' / 'mlflow.db').as_posix()}"
EXPERIMENT_NAME = "demand_forecast_backtest"
TARGET = "sales"

mlflow.set_tracking_uri(TRACKING_URI)
print("OK")


E:\JAIME\2023 3000\Estadística\18 Gemini CLI\ML2_Series_de_tiempo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK


## 1. Datos históricos con features

Se cargan `train_features.csv` (features del historial hasta 2017-09-30) y la escala naive
por serie se calculará por fold desde el propio train de cada ventana.

In [2]:
history = pd.read_csv(DATA_OUT / "train_features.csv", parse_dates=["date"])
print(f"Historial con features: {len(history):,} filas")
print(f"Rango: {history['date'].min().date()} a {history['date'].max().date()}")
print(f"Series: {history['store'].nunique()} tiendas x {history['item'].nunique()} items")


Historial con features: 255,600 filas
Rango: 2013-01-31 a 2017-09-30
Series: 10 tiendas x 15 items


## 2. Ventanas del backtest

Folds consecutivos de ~90 días. Train = todo lo anterior a la ventana.

In [3]:
folds_meta = walk_forward_folds(DEFAULT_TEST_ENDS)
for train_end, test_end in folds_meta:
        print(f"  train <= {str(train_end.date()):11s} | test {str((train_end + pd.Timedelta(days=1)).date())}..{str(test_end.date())}")
print(f"\n{len(folds_meta)} folds en total")


  train <= 2016-10-02  | test 2016-10-03..2016-12-31
  train <= 2016-12-31  | test 2017-01-01..2017-03-31
  train <= 2017-04-01  | test 2017-04-02..2017-06-30
  train <= 2017-07-02  | test 2017-07-03..2017-09-30

4 folds en total


C:\Users\Jaime\AppData\Local\Temp\ipykernel_5992\2894589908.py:3: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  print(f"  train <= {str(train_end.date()):11s} | test {str((train_end + pd.Timedelta(days=1)).date())}..{str(test_end.date())}")


## 3. Backtesting walk-forward

Se comparan los baselines y los dos mejores GBM del notebook 04 (LightGBM y XGBoost).
Cada modelo se re-entrena por fold y se acumulan las predicciones.

`walk_forward_backtest` devuelve:
- `folds`: métricas por (modelo, fold).
- `agg`: promedio online por modelo (columnas `<métrica>_online`).
- `pooled`: predicciones acumuladas por modelo para graficar.

In [4]:
configs = {
    name: all_configs()[name]
    for name in ["naive", "seasonal_naive", "mean", "lightgbm", "xgboost"]
}
features = [c for c in FEATURE_COLUMNS if c in history.columns]

folds, agg, pooled = walk_forward_backtest(
    history, configs, feature_names=features, target=TARGET
)
print("Backtest completado.")
folds[["modelo", "fold", "mase", "wape", "bias", "rel_bias_pct", "n_test"]].round(3)


Backtest completado.


,modelo,fold,mase,wape,bias,rel_bias_pct,n_test
0,naive,2016-12-31,1.02,19.49,0.22,0.40,13500
1,seasonal_naive,2016-12-31,0.86,16.16,1.36,2.42,13500
2,mean,2016-12-31,0.90,17.48,3.21,5.71,13500
3,lightgbm,2016-12-31,0.58,10.65,-0.11,-0.19,13500
4,xgboost,2016-12-31,0.58,10.69,-0.01,-0.02,13500
5,naive,2017-03-31,0.93,20.20,-0.09,-0.18,13500
6,seasonal_naive,2017-03-31,0.76,16.02,-0.89,-1.81,13500
7,mean,2017-03-31,0.74,16.36,-1.86,-3.79,13500
8,lightgbm,2017-03-31,0.54,11.28,0.23,0.47,13500
9,xgboost,2017-03-31,0.54,11.26,0.19,0.39,13500


## 4. Métricas online (promedio entre folds)

Mismo criterio que offline: MASE + WAPE con filtro de sesgo `<= 5%`.

In [5]:
agg.sort_values("mase_online")[["modelo", "mase_online", "wape_online", "mae_online", "bias_online", "rel_bias_pct_online"]].round(3)


,modelo,mase_online,wape_online,mae_online,bias_online,rel_bias_pct_online
0,lightgbm,0.60,10.18,6.27,-0.10,-0.11
4,xgboost,0.60,10.20,6.29,-0.12,-0.13
3,seasonal_naive,0.87,14.71,9.05,-0.01,-0.04
1,mean,0.89,15.64,9.65,0.04,0.05
2,naive,1.09,18.95,11.73,0.08,0.12


## 5. Registro en MLflow

Cada modelo queda como un run del experimento `demand_forecast_backtest` con sus métricas
online, parámetros y el CSV de detalle por fold (evidencia del pipeline online).

In [6]:
for name, cfg in configs.items():
    backtest_to_mlflow(
        cfg, folds, agg,
        experiment_name=EXPERIMENT_NAME,
        n_folds=len(folds_meta),
        horizon_days=90,
    )
print("Runs de backtest registrados en MLflow:", list(configs))


Runs de backtest registrados en MLflow: ['naive', 'seasonal_naive', 'mean', 'lightgbm', 'xgboost']


## 6. Pronóstico online acumulado

Agregado diario real vs predicho (acumulado de los 4 folds) para el mejor modelo online.

In [7]:
best = agg.sort_values("mase_online").iloc[0]["modelo"]
dfp = pooled[best]
daily = dfp.groupby("date")[["sales", "pred"]].sum()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily.index, daily["sales"], label="Real", marker="o", ms=2, lw=1.0)
ax.plot(daily.index, daily["pred"], label=f"Predicción ({best})", marker="x", ms=2, lw=1.0)
ax.set_title("Backtest walk-forward: pronóstico online agregado diario")
ax.set_xlabel("Fecha")
ax.set_ylabel("Unidades")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "backtest_online.png", dpi=150)
plt.show()


C:\Users\Jaime\AppData\Local\Temp\ipykernel_5992\3654540876.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Conclusiones

- El backtesting walk-forward confirma el ordenamiento offline: los **GBM superan a los
  baselines** también online (MASE < 1 y WAPE menor), con sesgo dentro del ±5%.
- El mejor modelo online es el mismo que offline (LightGBM o XGBoost según el fold), lo que
  valida que la selección no depende de un solo split.
- Queda evidencia en MLflow (`demand_forecast_backtest`) y el Código reusable en
  `src/models/backtesting.py`.
- Pendiente de Fase 4: MLflow remoto en DagsHub y modelo productivo `pyfunc` (notebook 06).
